In [40]:
# Export values from shapefiles to CSV for easier analysis and visualization in tools like Excel or Tableau.

from pathlib import Path
import re
from decimal import Decimal, ROUND_HALF_UP


import geopandas as gpd
import pandas as pd


# ---

def extract_year_from_name(name: str) -> int | str:
    m = YEAR_PATTERN.search(name)
    if m:
        return int(m.group('year'))
    return "Total all years"

def round_half_up_1(value: float) -> int | float:
    # Round half up to 1 decimal, but show 0 and 100 as integers
    rounded = Decimal(str(value)).quantize(Decimal("0.1"), rounding=ROUND_HALF_UP)
    if rounded == 0 or rounded == Decimal("100.0"):
        return int(rounded)
    return float(rounded)

# ---

MAP = "output_Inundation_Zones" # "output_eflows" # output_NearBottomFV

# Output path location for shapefile data created 
OUTPUT_DIR: Path = Path(rf'example_output_data_20260223/{MAP}')

# Helper: extract year from filename (e.g., "..._2020.shp" or "...2020-12-31.shp")
YEAR_PATTERN: re.Pattern = re.compile(r'(?P<year>19\d{2}|20\d{2})')

# Find all shapefiles in output directory (recursively, in case of subfolders)
SHAPFILES_LIST: list[Path] = list(OUTPUT_DIR.rglob("*.shp"))
if not SHAPFILES_LIST:
    print(f"⚠️ No shapefiles found under: {OUTPUT_DIR.resolve()}")
print(f"🔍 Found {len(SHAPFILES_LIST)} shapefile(s) under: {OUTPUT_DIR.resolve()}")



🔍 Found 7 shapefile(s) under: C:\Users\kelde_sn\REPOS\react\notebooks\post-processing\example_output_data_20260223\output_Inundation_Zones


In [41]:
# Collect all values from column 'guilds_ADV' across shapefiles in SHAPFILES_LIST
target_col = "Zone 5"

all_values = []
rows = []

for shapefile in SHAPFILES_LIST:
    try:
        _gdf = gpd.read_file(shapefile)
    except Exception as e:
        print(f"⚠️ Could not read {shapefile}: {e}")
        continue

    if target_col not in _gdf.columns:
        print(f"ℹ️ Column '{target_col}' not found in: {shapefile.name}")
        continue

    s = _gdf[target_col]
    all_values.append(s)

    for v, c in s.value_counts(dropna=False).items():
        rows.append({
            "file": str(shapefile),
            "year": extract_year_from_name(shapefile.stem),
            target_col: v,
            "count": int(c)
        })

if all_values:
    all_values_concat = pd.concat(all_values, ignore_index=True)
    unique_values = sorted(all_values_concat.dropna().unique().tolist())
    counts_all = (
        all_values_concat.value_counts(dropna=False)
        .rename_axis(target_col)
        .reset_index(name="count")
    )
    values_by_file = pd.DataFrame(rows)

    print(f"✅ Unique non-null values for '{target_col}':")
    print(unique_values)

    print(f"\n✅ Counts across all shapefiles for '{target_col}':")
    print(counts_all.to_string(index=False))

    print(f"\n✅ Counts per file for '{target_col}':")
    print(values_by_file.to_string(index=False))
else:
    print(f"ℹ️ No values found for column '{target_col}' in SHAPFILES_LIST.")

✅ Unique non-null values for 'Zone 5':
[0.0, 0.5, 1.0]

✅ Counts across all shapefiles for 'Zone 5':
 Zone 5   count
    0.0 1229042
    0.5    4760
    1.0    1250

✅ Counts per file for 'Zone 5':
                                                                                                                                        file            year  Zone 5  count
                                                                      example_output_data_20260223\output_Inundation_Zones\complete_file.shp Total all years     0.0 614521
                                                                      example_output_data_20260223\output_Inundation_Zones\complete_file.shp Total all years     0.5   2380
                                                                      example_output_data_20260223\output_Inundation_Zones\complete_file.shp Total all years     1.0    625
example_output_data_20260223\output_Inundation_Zones\results_REACT_WFLOW_inundation_zones_2018\results_REACT_WFLOW

In [32]:
# Custom example for these specific variables, but can be adapted to other variables as needed.

# List with variables that need to be exported to shapefile. Note, these should only contain faces as axes (no time or depth etc).
VARS_EFLOWS: list = [
'low_flow',
'inter_flow',
'high_flow',
]

# ---

# Collect per-file, per-variable counts
rows: list[dict[str, int | str | None]] = []

for shapefile in SHAPFILES_LIST:
    try:
        gdf = gpd.read_file(shapefile)
    except Exception as e:
        print(f"⚠️ Could not read {shapefile}: {e}")
        continue

    # Try to infer year from filename
    year = extract_year_from_name(shapefile.stem)

    # For each target variable, if it exists in this shapefile, count values
    for var in VARS_EFLOWS:
        if var not in gdf.columns:
            # Not all shapefiles necessarily contain all variables; skip quietly
            continue

        sel = gdf[var]

        # Numeric class values aligned to geometry
        sel_num = pd.to_numeric(sel, errors='coerce')

        # Use projected CRS for area (m² -> km²)
        gdf_area = gdf
        if gdf_area.crs is None:
            gdf_area = gdf_area.set_crs("EPSG:4326")
        if gdf_area.crs.to_epsg() != 28992:
            gdf_area = gdf_area.to_crs("EPSG:28992")

        tmp = pd.DataFrame({
            "class": sel_num,
            "area_km2": gdf_area.geometry.area / 1_000_000.0
        }, index=gdf.index).dropna(subset=["class"])

        tmp["class"] = tmp["class"].round().astype(int)

        # Counts
        counts = tmp["class"].value_counts()
        count_neg_1 = int(counts.get(-1, 0))
        count_1 = int(counts.get(1, 0))
        count_2 = int(counts.get(2, 0))
        count_total = int(len(tmp))

        # Areas (km²) - raw
        area_by_class = tmp.groupby("class")["area_km2"].sum()
        km2_neg_1_raw = float(area_by_class.get(-1, 0.0))
        km2_1_raw = float(area_by_class.get(1, 0.0))
        km2_2_raw = float(area_by_class.get(2, 0.0))
        km2_total_raw = float(tmp["area_km2"].sum())

        # Areas (km²) - rounded (half up, 1 decimal)
        km2_neg_1 = round_half_up_1(km2_neg_1_raw)
        km2_1 = round_half_up_1(km2_1_raw)
        km2_2 = round_half_up_1(km2_2_raw)
        km2_total = round_half_up_1(km2_total_raw)

        # Percentages by count (%) - rounded
        pct_neg_1 = round_half_up_1((count_neg_1 / count_total * 100.0) if count_total else 0.0)
        pct_1 = round_half_up_1((count_1 / count_total * 100.0) if count_total else 0.0)
        pct_2 = round_half_up_1((count_2 / count_total * 100.0) if count_total else 0.0)

        # Percentages by area (% of total km²) - rounded
        pct_km2_neg_1 = round_half_up_1((km2_neg_1_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)
        pct_km2_1 = round_half_up_1((km2_1_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)
        pct_km2_2 = round_half_up_1((km2_2_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)

        rows.append({
            "variable": var,
            "year": year,  # Also the summed value of all years, as "Total all years"
            "file": str(shapefile),
            "area_masked_out_count": count_neg_1,
            "area_bad_flow_count":  count_1,
            "area_good_flow_count":  count_2,
            "area_total_count": count_total,
            "area_masked_out_km2": km2_neg_1,
            "area_bad_flow_km2":  km2_1,
            "area_good_flow_km2":  km2_2,
            "area_total_km2": km2_total,
            "pct_masked_out_count": pct_neg_1,
            "pct_bad_flow_count": pct_1,
            "pct_good_flow_count": pct_2,
            "pct_masked_out_km2": pct_km2_neg_1,
            "pct_bad_flow_km2": pct_km2_1,
            "pct_good_flow_km2": pct_km2_2
        })

summary_df = pd.DataFrame(rows)

if not summary_df.empty:
    # Per-year summary (summing across files for same var-year)
    per_year = (summary_df
                .groupby(["variable", "year"], dropna=False)[
                    ["area_masked_out_count", "area_bad_flow_count", "area_good_flow_count", "area_total_count",
                     "area_masked_out_km2", "area_bad_flow_km2", "area_good_flow_km2", "area_total_km2",
                     "pct_masked_out_count", "pct_bad_flow_count", "pct_good_flow_count",
                     "pct_masked_out_km2", "pct_bad_flow_km2", "pct_good_flow_km2"]]
                .sum()
                .reset_index()
               ).sort_values(["variable", "year"])

    # Write outputs
    out_csv_per_year = OUTPUT_DIR / "summary_counts_area_pct_per_year.csv"
    per_year.to_csv(out_csv_per_year, sep=';', index=False)


    # Display a small preview
    print("\n✅ Summary written:")
    print(f"  - {out_csv_per_year}")
    print("\nPer-year preview:")
    print(per_year.to_string(index=False))

else:
    print("ℹ️ No variable counts were collected. Check that shapefiles contain the expected fields: "
          f"{VARS_EFLOWS} and that files exist under {OUTPUT_DIR.resolve()}.")



✅ Summary written:
  - example_output_data_20260223\output_NearBottomFV\summary_counts_area_pct_per_year.csv

Per-year preview:
  variable            year  area_masked_out_count  area_bad_flow_count  area_good_flow_count  area_total_count  area_masked_out_km2  area_bad_flow_km2  area_good_flow_km2  area_total_km2  pct_masked_out_count  pct_bad_flow_count  pct_good_flow_count  pct_masked_out_km2  pct_bad_flow_km2  pct_good_flow_km2
guilds_ADV            2018                      0                  268                     0              2584                    0               22.5                 0.0           217.3                     0                10.4                    0                   0              10.3                  0
guilds_ADV            2019                      0                  327                     0              2584                    0               27.4                 0.0           217.3                     0                12.7                    0        

In [45]:
# Custom example for these specific variables, but can be adapted to other variables as needed.

# List with variables that need to be exported to shapefile. Note, these should only contain faces as axes (no time or depth etc).
VARS_INUNDATION_ZONES: list = [
'Zone 5',
]

# ---

# Collect per-file, per-variable counts
rows: list[dict[str, int | str | None]] = []

for shapefile in SHAPFILES_LIST:
    try:
        gdf = gpd.read_file(shapefile)
    except Exception as e:
        print(f"⚠️ Could not read {shapefile}: {e}")
        continue

    # Try to infer year from filename
    year = extract_year_from_name(shapefile.stem)

    # For each target variable, if it exists in this shapefile, count values
    for var in VARS_INUNDATION_ZONES:
        if var not in gdf.columns:
            # Not all shapefiles necessarily contain all variables; skip quietly
            continue

        sel = gdf[var]

        # Numeric class values aligned to geometry
        sel_num = pd.to_numeric(sel, errors='coerce')

        # Use projected CRS for area (m² -> km²)
        gdf_area = gdf
        if gdf_area.crs is None:
            gdf_area = gdf_area.set_crs("EPSG:4326")
        if gdf_area.crs.to_epsg() != 28992:
            gdf_area = gdf_area.to_crs("EPSG:28992")

        tmp = pd.DataFrame({
            "class": sel_num,
            "area_km2": gdf_area.geometry.area / 1_000_000.0
        }, index=gdf.index).dropna(subset=["class"])

        # Keep fractional classes (e.g. 0.5); do not cast to int
        tmp["class"] = pd.to_numeric(tmp["class"], errors="coerce").round(1)

        # Counts
        counts = tmp["class"].value_counts()
        count_0 = int(counts.get(0, 0))
        count_0_5 = int(counts.get(0.5, 0))
        count_1 = int(counts.get(1, 0))
        count_total = int(len(tmp))

        # Areas (km²) - raw
        area_by_class = tmp.groupby("class")["area_km2"].sum()
        km2_0_raw = float(area_by_class.get(0, 0.0))
        km2_0_5_raw = float(area_by_class.get(0.5, 0.0))
        km2_1_raw = float(area_by_class.get(1, 0.0))
        km2_total_raw = float(tmp["area_km2"].sum())

        # Areas (km²) - rounded (half up, 1 decimal)
        km2_0 = round_half_up_1(km2_0_raw)
        km2_0_5 = round_half_up_1(km2_0_5_raw)
        km2_1 = round_half_up_1(km2_1_raw)
        km2_total = round_half_up_1(km2_total_raw)

        # Percentages by count (%) - rounded
        pct_0 = round_half_up_1((count_0 / count_total * 100.0) if count_total else 0.0)
        pct_0_5 = round_half_up_1((count_0_5 / count_total * 100.0) if count_total else 0.0)
        pct_1 = round_half_up_1((count_1 / count_total * 100.0) if count_total else 0.0)

        # Percentages by area (% of total km²) - rounded
        pct_km2_0 = round_half_up_1((km2_0_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)
        pct_km2_0_5 = round_half_up_1((km2_0_5_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)
        pct_km2_1 = round_half_up_1((km2_1_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)

        rows.append({
            "variable": var,
            "year": year,  # Also the summed value of all years, as "Total all years"
            "file": str(shapefile),
            "area_0_count": count_0,
            "area_0_5_count": count_0_5,
            "area_1_count":  count_1,
            "area_total_count": count_total,
            "area_0_km2": km2_0,
            "area_0_5_km2": km2_0_5,
            "area_1_km2":  km2_1,
            "area_total_km2": km2_total,
            "pct_0_count": pct_0,
            "pct_0_5_count": pct_0_5,
            "pct_1_count": pct_1,
            "pct_0_km2": pct_km2_0,
            "pct_0_5_km2": pct_km2_0_5,
            "pct_1_km2": pct_km2_1
        })

summary_df = pd.DataFrame(rows)

if not summary_df.empty:
    # Per-year summary (summing across files for same var-year)
    per_year = (summary_df
                .groupby(["variable", "year"], dropna=False)[
                    ["area_0_count", "area_0_5_count", "area_1_count", "area_total_count",
                     "area_0_km2", "area_0_5_km2", "area_1_km2", "area_total_km2",
                     "pct_0_count", "pct_0_5_count", "pct_1_count",
                     "pct_0_km2", "pct_0_5_km2", "pct_1_km2"]]
                .sum()
                .reset_index()
               ).sort_values(["variable", "year"])

    # Write outputs
    out_csv_per_year = OUTPUT_DIR / "summary_counts_area_pct_per_year.csv"
    per_year.to_csv(out_csv_per_year, sep=';', index=False)


    # Display a small preview
    print("\n✅ Summary written:")
    print(f"  - {out_csv_per_year}")
    print("\nPer-year preview:")
    print(per_year.to_string(index=False))

else:
    print("ℹ️ No variable counts were collected. Check that shapefiles contain the expected fields: "
          f"{VARS_INUNDATION_ZONES} and that files exist under {OUTPUT_DIR.resolve()}.")


✅ Summary written:
  - example_output_data_20260223\output_Inundation_Zones\summary_counts_area_pct_per_year.csv

Per-year preview:
variable            year  area_0_count  area_0_5_count  area_1_count  area_total_count  area_0_km2  area_0_5_km2  area_1_km2  area_total_km2  pct_0_count  pct_0_5_count  pct_1_count  pct_0_km2  pct_0_5_km2  pct_1_km2
  Zone 5            2018        102518             364            39            102921      8622.7          30.6         3.3          8656.6         99.6            0.4          0.0       99.6          0.4        0.0
  Zone 5            2019        102550             302            69            102921      8625.4          25.4         5.8          8656.6         99.6            0.3          0.1       99.6          0.3        0.1
  Zone 5            2020        102379             445            97            102921      8611.0          37.4         8.2          8656.6         99.5            0.4          0.1       99.5          0.4        0.1

In [33]:
# Custom example for these specific variables, but can be adapted to other variables as needed.

# List with variables that need to be exported to shapefile. Note, these should only contain faces as axes (no time or depth etc).
VARS_NEARBOTTOMFV: list = [
'guilds_ADV',
]


# ---

# Collect per-file, per-variable counts
rows: list[dict[str, int | str | None]] = []

for shapefile in SHAPFILES_LIST:
    try:
        gdf = gpd.read_file(shapefile)
    except Exception as e:
        print(f"⚠️ Could not read {shapefile}: {e}")
        continue

    # Try to infer year from filename
    year = extract_year_from_name(shapefile.stem)

    # For each target variable, if it exists in this shapefile, count values
    for var in VARS_NEARBOTTOMFV:
        if var not in gdf.columns:
            # Not all shapefiles necessarily contain all variables; skip quietly
            continue

        sel = gdf[var]

        # Numeric class values aligned to geometry
        sel_num = pd.to_numeric(sel, errors='coerce')

        # Use projected CRS for area (m² -> km²)
        gdf_area = gdf
        if gdf_area.crs is None:
            gdf_area = gdf_area.set_crs("EPSG:4326")
        if gdf_area.crs.to_epsg() != 28992:
            gdf_area = gdf_area.to_crs("EPSG:28992")

        tmp = pd.DataFrame({
            "class": sel_num,
            "area_km2": gdf_area.geometry.area / 1_000_000.0
        }, index=gdf.index).dropna(subset=["class"])

        tmp["class"] = tmp["class"].round().astype(int)

        # Counts
        counts = tmp["class"].value_counts()
        count_1 = int(counts.get(1, 0))
        count_2 = int(counts.get(2, 0))
        count_3 = int(counts.get(3, 0))
        count_4 = int(counts.get(4, 0))
        count_5 = int(counts.get(5, 0))
        count_9 = int(counts.get(9, 0))
        count_10 = int(counts.get(10, 0))
        count_11 = int(counts.get(11, 0))
        count_total = int(len(tmp))

        # Areas (km²) - raw
        area_by_class = tmp.groupby("class")["area_km2"].sum()
        km2_1_raw = float(area_by_class.get(1, 0.0))
        km2_2_raw = float(area_by_class.get(2, 0.0))
        km2_3_raw = float(area_by_class.get(3, 0.0))
        km2_4_raw = float(area_by_class.get(4, 0.0))
        km2_5_raw = float(area_by_class.get(5, 0.0))
        km2_9_raw = float(area_by_class.get(9, 0.0))
        km2_10_raw = float(area_by_class.get(10, 0.0))
        km2_11_raw = float(area_by_class.get(11, 0.0))
        km2_total_raw = float(tmp["area_km2"].sum())

        # Areas (km²) - rounded (half up, 1 decimal)
        km2_1 = round_half_up_1(km2_1_raw)
        km2_2 = round_half_up_1(km2_2_raw)
        km2_3 = round_half_up_1(km2_3_raw)
        km2_4 = round_half_up_1(km2_4_raw)
        km2_5 = round_half_up_1(km2_5_raw)
        km2_9 = round_half_up_1(km2_9_raw)
        km2_10 = round_half_up_1(km2_10_raw)
        km2_11 = round_half_up_1(km2_11_raw)
        km2_total = round_half_up_1(km2_total_raw)

        # Percentages by count (%) - rounded
        pct_1 = round_half_up_1((count_1 / count_total * 100.0) if count_total else 0.0)
        pct_2 = round_half_up_1((count_2/ count_total * 100.0) if count_total else 0.0)
        pct_3 = round_half_up_1((count_3 / count_total * 100.0) if count_total else 0.0)
        pct_4 = round_half_up_1((count_4 / count_total * 100.0) if count_total else 0.0)
        pct_5 = round_half_up_1((count_5 / count_total * 100.0) if count_total else 0.0)
        pct_9 = round_half_up_1((count_9 / count_total * 100.0) if count_total else 0.0)
        pct_10 = round_half_up_1((count_10 / count_total * 100.0) if count_total else 0.0)
        pct_11 = round_half_up_1((count_11 / count_total * 100.0) if count_total else 0.0)

        # Percentages by area (% of total km²) - rounded
        pct_km2_1 = round_half_up_1((km2_1_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)
        pct_km2_2 = round_half_up_1((km2_2_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)
        pct_km2_3 = round_half_up_1((km2_3_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)
        pct_km2_4 = round_half_up_1((km2_4_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)
        pct_km2_5 = round_half_up_1((km2_5_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)
        pct_km2_9 = round_half_up_1((km2_9_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)
        pct_km2_10 = round_half_up_1((km2_10_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)
        pct_km2_11 = round_half_up_1((km2_11_raw / km2_total_raw * 100.0) if km2_total_raw else 0.0)

        rows.append({
            "variable": var,
            "year": year,  # Also the summed value of all years, as "Total all years"
            "file": str(shapefile),

            "area_cat_1_count": count_1,
            "area_cat_2_count": count_2,
            "area_cat_3_count": count_3,
            "area_cat_4_count": count_4,
            "area_cat_5_count": count_5,
            "area_cat_9_count": count_9,
            "area_cat_10_count": count_10,
            "area_cat_11_count": count_11,
            "area_total_count": count_total,

            "area_cat_1_km2": km2_1,
            "area_cat_2_km2": km2_2,
            "area_cat_3_km2": km2_3,
            "area_cat_4_km2": km2_4,
            "area_cat_5_km2": km2_5,
            "area_cat_9_km2": km2_9,
            "area_cat_10_km2": km2_10,
            "area_cat_11_km2": km2_11,
            "area_total_km2": km2_total,

            "pct_cat_1_count": pct_1,
            "pct_cat_2_count": pct_2,
            "pct_cat_3_count": pct_3,
            "pct_cat_4_count": pct_4,
            "pct_cat_5_count": pct_5,
            "pct_cat_9_count": pct_9,
            "pct_cat_10_count": pct_10,
            "pct_cat_11_count": pct_11,

            "pct_cat_1_km2": pct_km2_1,
            "pct_cat_2_km2": pct_km2_2,
            "pct_cat_3_km2": pct_km2_3,
            "pct_cat_4_km2": pct_km2_4,
            "pct_cat_5_km2": pct_km2_5,
            "pct_cat_9_km2": pct_km2_9,
            "pct_cat_10_km2": pct_km2_10,
            "pct_cat_11_km2": pct_km2_11
        })

summary_df = pd.DataFrame(rows)

if not summary_df.empty:
    # Per-year summary (summing across files for same var-year)
    per_year = (summary_df
                .groupby(["variable", "year"], dropna=False)[
                    ["area_cat_1_count", "area_cat_2_count", "area_cat_3_count", "area_cat_4_count", "area_cat_5_count", "area_cat_9_count", "area_cat_10_count", "area_cat_11_count", "area_total_count",
                     "area_cat_1_km2", "area_cat_2_km2", "area_cat_3_km2", "area_cat_4_km2", "area_cat_5_km2", "area_cat_9_km2", "area_cat_10_km2", "area_cat_11_km2", "area_total_km2",
                     "pct_cat_1_count", "pct_cat_2_count", "pct_cat_3_count", "pct_cat_4_count", "pct_cat_5_count", "pct_cat_9_count", "pct_cat_10_count", "pct_cat_11_count",
                     "pct_cat_1_km2", "pct_cat_2_km2", "pct_cat_3_km2", "pct_cat_4_km2", "pct_cat_5_km2", "pct_cat_9_km2", "pct_cat_10_km2", "pct_cat_11_km2"]]
                .sum()
                .reset_index()
               ).sort_values(["variable", "year"])

    # Write outputs
    out_csv_per_year = OUTPUT_DIR / "summary_counts_area_pct_per_year.csv"
    per_year.to_csv(out_csv_per_year, sep=';', index=False)


    # Display a small preview
    print("\n✅ Summary written:")
    print(f"  - {out_csv_per_year}")
    print("\nPer-year preview:")
    print(per_year.to_string(index=False))

else:
    print("ℹ️ No variable counts were collected. Check that shapefiles contain the expected fields: "
          f"{VARS_NEARBOTTOMFV} and that files exist under {OUTPUT_DIR.resolve()}.")



✅ Summary written:
  - example_output_data_20260223\output_NearBottomFV\summary_counts_area_pct_per_year.csv

Per-year preview:
  variable            year  area_cat_1_count  area_cat_2_count  area_cat_3_count  area_cat_4_count  area_cat_5_count  area_cat_9_count  area_cat_10_count  area_cat_11_count  area_total_count  area_cat_1_km2  area_cat_2_km2  area_cat_3_km2  area_cat_4_km2  area_cat_5_km2  area_cat_9_km2  area_cat_10_km2  area_cat_11_km2  area_total_km2  pct_cat_1_count  pct_cat_2_count  pct_cat_3_count  pct_cat_4_count  pct_cat_5_count  pct_cat_9_count  pct_cat_10_count  pct_cat_11_count  pct_cat_1_km2  pct_cat_2_km2  pct_cat_3_km2  pct_cat_4_km2  pct_cat_5_km2  pct_cat_9_km2  pct_cat_10_km2  pct_cat_11_km2
guilds_ADV            2018               268                 0               956               470                 0               223                667                  0              2584            22.5             0.0            80.4            39.4             0.0    